
# Tutorial — Interface de modelos de série temporal (`SatelliteUI`)

Modelos **satélite** ligam a série temporal agregada de um parâmetro de risco de um segmento —
taxa de *default* (PD), severidade (LGD), fator de conversão (CCF) — às **variáveis
macroeconômicas**, e respondem à pergunta que a área toda faz: *dado um cenário, quanto vale esse
parâmetro nos próximos trimestres?* É o **fator prospectivo**: a projeção **condicional a cenário**
que alimenta provisionamento, estresse, capital e planejamento.

O tutorial [`09_tutorial_modelos_econometricos.ipynb`](09_tutorial_modelos_econometricos.ipynb)
ensina a **API** desse subpacote (`yggdrasil.credit_risk.econometric`): transformações, ARDL, fator
$Z$, busca champion-challenger, cenários, `run_study`. **Este tutorial não repete a teoria** — ele
ensina a **interface** que roda tudo aquilo sem exigir que ninguém decore a API.

`SatelliteUI` é um *workbench* de **7 abas** em `ipywidgets`, no mesmo padrão do `TreeSegmenterUI`
e do `ModelSegmenterUI` (tutoriais 04, 05 e 06):

| Aba | Em uma frase |
|-----|--------------|
| **① Série** | o que entrou: a série no nível e na escala do *link*, as macro disponíveis e o ritual de estacionariedade |
| **② Especificação** | qual modelo, quais candidatas, com que **sinal esperado** e defasagem — mais o ajuste único de feedback rápido |
| **③ Seleção** | a busca champion-challenger: ranking, **motivo de cada descarte** e a especificação que você adota |
| **④ Diagnóstico** | a bateria sobre o resíduo do modelo vigente, com placar por família de teste e o que fazer quando reprova |
| **⑤ Cenários & Projeção** | três caminhos para montar o cenário, a projeção **em leque** e a curva única ponderada |
| **⑥ Backtest** | a outra metade da projeção: a **banda** cobre o que promete? (Kupiec e Christoffersen) |
| **⑦ Exportar** | relatório HTML, *run* no MLflow, o JSON da configuração e as tabelas em CSV |

> **Como este notebook funciona.** Um notebook estático não registra cliques. Cada aba é explicada
> e, em seguida, **reproduzida em código**: as células mexem nos mesmos widgets que você mexeria na
> tela e chamam `ui.btn_*.click()`, que é literalmente o que o botão faz. O estado da interface
> fica igual ao de quem clicou — e o que sobra no fim (`ui.fit_`, `ui.projection_`, …) é o que a
> seção 6 leva de volta para o código.

> **Dependências:** `ipywidgets` (extra `[ui]`) para a interface e `statsmodels` + `arch`
> (extra `[econometric]`) para os modelos — `pip install "yggdrasil-project[ui,econometric]"`, ou
> `pip install -e ".[ui,econometric]"` a partir do repositório. O núcleo de `credit_risk` não
> exige nenhum dos dois: o subpacote é carregado sob demanda.


## 0. Setup

Nada além do import: a interface só precisa de `ipywidgets` na sessão e de uma série + macro em
memória. `%matplotlib inline` porque a UI devolve as figuras já embutidas como imagem.

In [ ]:
# --- Bootstrap: torna o pacote `yggdrasil` importável a partir do repositório,
# sem `pip install`. Procura a raiz do repo (a pasta que contém `yggdrasil/`) por
# vários âncoras — o caminho do próprio notebook (VS Code expõe `__vsc_ipynb_file__`),
# o diretório atual, os diretórios do sys.path e, no Databricks, o caminho via
# dbutils — subindo até achá-la, e a insere no sys.path. Cobre Jupyter/VS Code
# local e Databricks; se o pacote já estiver importável, é inócuo.
import sys
from pathlib import Path


def _find_yggdrasil_root():
    _anchors = []
    for _n in ("__vsc_ipynb_file__", "__file__", "__session__"):
        _v = globals().get(_n)
        if _v:
            _anchors.append(Path(str(_v)))
    _anchors.append(Path.cwd())
    _anchors += [Path(_p) for _p in sys.path if _p not in ("", ".")]
    for _a in _anchors:
        try:
            _a = _a.resolve()
        except Exception:
            continue
        for _b in (_a, *_a.parents):
            if (_b / "yggdrasil" / "__init__.py").is_file():
                return _b
    try:  # fallback Databricks: caminho do próprio notebook
        _nbp = (dbutils.notebook.entry_point.getDbutils()  # noqa: F821
                .notebook().getContext().notebookPath().get())
        for _pref in ("/Workspace", ""):
            for _b in Path(_pref + _nbp).parents:
                if (_b / "yggdrasil" / "__init__.py").is_file():
                    return _b
    except Exception:
        pass
    return None


_ygg_root = _find_yggdrasil_root()
if _ygg_root and str(_ygg_root) not in sys.path:
    sys.path.insert(0, str(_ygg_root))

In [ ]:
%matplotlib inline
import os
os.environ.setdefault("MLFLOW_ALLOW_FILE_STORE", "true")   # MLflow local (seção 9)

import warnings; warnings.filterwarnings("ignore")          # avisos de convergência do statsmodels
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.max_columns", 40)

import yggdrasil.credit_risk.econometric as E
from yggdrasil.credit_risk.econometric import SatelliteUI
print("econometric — API com", len(E.__all__), "símbolos públicos")


## 1. Os dados: uma série do parâmetro + um painel macro

O contrato da interface é curto:

- **a série do parâmetro** — uma taxa em $[0, 1]$ por período, de um **segmento homogêneo**, com
  índice temporal (`DatetimeIndex` ou `PeriodIndex`);
- **a macro** — um `DataFrame` de variáveis numéricas no **mesmo calendário** (pode ter períodos a
  mais no fim; não pode faltar período da série).

Sem base em mãos, o **estudo de referência** dá os dois na hora: séries sintéticas de processo
gerador conhecido (PD, LGD e CCF do mesmo segmento) dirigidas por uma macro com recessão e evento.
É o mesmo `make_reference_study()` do tutorial 09 — e é exatamente o que o botão *Carregar estudo de
referência*, na aba ① Série, faz por dentro.

In [ ]:
est = E.make_reference_study(n_periods=96, seed=7)   # 8 anos mensais
macro = est.macro
serie_rs = est.pd.series          # RiskSeries: valores + kind + segmento + frequência

print("Macro:", list(macro.columns), "| períodos:", len(macro))
print(f"Série: kind={serie_rs.kind} · segmento={serie_rs.segment} · freq={serie_rs.frequency} "
      f"· média={serie_rs.values.mean():.4f}")
macro.tail(3)


### 1.1. Plugando os **seus** dados

No caso real a série vem de uma consulta (contratos em *default* sobre a base elegível por safra,
severidade realizada por safra de recuperação, etc.) e a macro vem do painel econômico. A interface
aceita os dois como objetos **crus do pandas** — não é preciso construir nada da biblioteca:

```python
ui = SatelliteUI(
    series=minha_serie,       # pd.Series de taxa em [0,1], índice temporal
    macro=minha_macro,        # pd.DataFrame de macro, mesmo calendário
    kind="pd",                # "pd", "lgd" ou "ccf"
    segment="cartao_rotativo",
    problem_label="PD",       # rótulo do parâmetro nos títulos (default: o kind)
    name="pd_cartao_rotativo",
)
```

Se você já trabalha com `RiskSeries` (o objeto do tutorial 09), passe-o direto — o `kind`, o
segmento e a frequência vêm dele. Abaixo simulamos o caso real: extraímos do estudo de referência
uma `pd.Series` pura, que é o que sairia da sua consulta.

In [ ]:
# O que sairia da sua consulta: uma taxa por período, índice temporal, e a macro ao lado.
taxa = pd.Series(serie_rs.values.to_numpy(), index=serie_rs.values.index, name="pd_observada")

print(type(taxa).__name__, "| índice:", type(taxa.index).__name__,
      "|", taxa.index.min().date(), "→", taxa.index.max().date())
print("taxa em [0,1]:", bool(taxa.between(0, 1).all()),
      "| calendário alinhado com a macro:", bool(taxa.index.equals(macro.index)))
taxa.head(3)


## 2. Abrindo a interface

Instancie e deixe o objeto como **última linha** da célula — o Jupyter chama `_ipython_display_()`
e desenha o painel (alternativas: `ui.display()` ou `display(ui.panel)`).

Daqui em diante **o trabalho é na tela**. O notebook segue apenas comentando o que olhar em cada
aba e reproduzindo os cliques em código, para que você possa ler o tutorial inteiro sem uma sessão
interativa aberta.

> **Sem dados em mãos?** `SatelliteUI()` abre vazia, com a aba ① pronta para o botão *Carregar
> estudo de referência* — dá para percorrer a ferramenta inteira antes de ter a série do segmento.
> E `ui.set_data(outra_serie, outra_macro, kind="lgd")` troca os dados sem fechar a interface
> (descartando ajuste, busca e projeção, que pertenciam aos dados antigos).

Duas coisas úteis no alto do painel: o **🌙 tema escuro** e o **☕ manter cluster ativo** — este
último dispara um job mínimo a cada 2 minutos para o cluster do Databricks não desligar por
inatividade no meio de uma busca longa (fora do Spark é inócuo). Logo abaixo há a **barra de
estado** (o que já existe na sessão) e, no rodapé, o **console** com o histórico das ações.

In [ ]:
ui = SatelliteUI(
    series=taxa,                     # a pd.Series crua da seção 1.1
    macro=macro,
    kind="pd",                       # troque para "lgd" ou "ccf" — a tela é a mesma
    segment="segmento_referencia",
    problem_label="PD",              # rótulo neutro: a interface serve PD, LGD e CCF
    name="pd_segmento_referencia",
)

ui   # exibe o workbench de 7 abas


## 3. Aba ① **Série** — o que entrou, e ele é modelável?

**O que olhar.** Três coisas, nesta ordem:

1. **O cabeçalho e os dois gráficos.** À esquerda a série no **nível** (a taxa como ela é), à
   direita na **escala do link** (logit/probit). Os modelos estimam na escala do link — é ela que
   leva a taxa de $[0,1]$ para toda a reta e garante que a projeção nunca saia do intervalo. A
   série **transformada** é a que precisa parecer bem-comportada.
2. **As macro disponíveis.** Selecione algumas (Ctrl/Shift) e desenhe: variáveis muito parecidas
   entre si disputam o mesmo papel e inflam o VIF depois.
3. **A estacionariedade** — o passo que todo mundo pula. **ADF** tem H0 = raiz unitária (p pequeno
   ⇒ estacionária); **KPSS** tem a nula **oposta**; **Phillips-Perron** é o desempate. A coluna
   **I(d)** diz quantas diferenças a série precisaria para os dois concordarem.

**A decisão.** Se o alvo e as candidatas são todos **I(1)**, regredir nível contra nível produz
**regressão espúria** — R² alto e relação inexistente. Aí as saídas são: modelar na escala do link
(que costuma resolver, e é o padrão), trabalhar em diferenças, ou tratar a relação como equilíbrio
de longo prazo (cointegração/VECM, que está na API do tutorial 09, não nesta tela).

In [ ]:
# = selecionar macros e clicar "Desenhar macros" (a figura vai para o painel da aba ①)
ui.sel_macro_plot.value = ("desemprego", "juros")
ui.btn_macro_plot.click()

# = clicar "Rodar testes de estacionariedade"
ui.fl_alpha_estac.value = 0.05
ui.cb_estac_todas.value = True     # inclui todas as macro, não só as candidatas
ui.btn_estac.click()

ui.stationarity_                   # alvo (nível e link) + cada macro, com a ordem I(d)


## 4. Aba ② **Especificação** — modelo, sinais esperados e defasagens

**O que olhar.**

- **Modelo.** O dropdown traz o catálogo com a ajuda de cada um: **ARDL** (o padrão — inércia +
  macro defasada), **fator $Z$ de Vasicek** (ponte com o motor de capital, só para PD),
  **regressão beta** e **fractional logit** (frações, com e sem massa em 0/1), **ARIMA/ARIMAX** e
  os **ingênuos** (passeio aleatório, média histórica, sazonal). Os campos específicos —
  $\rho$, nível de longo prazo, ordem $(p,d,q)$ — aparecem e somem conforme a escolha.
- **Candidatas e sinais esperados.** É o coração da tela. Para cada macro: entra na busca? qual o
  **sinal econômico esperado** (`+1` a alta piora o parâmetro, `−1` melhora, `livre` quando a
  teoria não decide)? qual a **defasagem** para o ajuste único? O botão *Sugerir sinais* preenche
  pela família da variável — **revise sempre**, é sugestão, não veredito.
- **Grade e regras.** Defasagens que a busca varre, ordens AR, máximo de variáveis por
  especificação, teto de VIF, critério de ranking, horizonte e mínimo de treino da validação.
- **Ajustar agora.** Antes de gastar uma busca inteira, ajuste **uma** especificação e olhe sinais,
  p-valores e AIC/BIC. A coluna **coerência** compara o sinal estimado com o que você declarou.

**A decisão.** O sinal esperado é o **filtro duro** da coerência econômica: uma especificação com o
sinal trocado é desqualificada por melhor que seja o ajuste, porque sob estresse a projeção iria na
direção errada. Declarar sinal é assumir uma hipótese econômica — e é a parte do modelo que a
validação independente vai perguntar primeiro.

In [ ]:
# = escolher o modelo e preencher a matriz de candidatas na aba ② Especificação
ui.dd_model.value = "ardl"          # veja o catálogo em [rot for rot, _ in ui.dd_model.options]
ui.dd_link.value = "logit"
ui.dd_trend.value = "c"

# {variável: (sinal esperado, defasagem do "Ajustar agora")}
candidatas = {"desemprego": (+1, 1), "renda": (-1, 0), "juros": (+1, 2)}
for var, cb in ui._sign_cbs.items():          # a matriz de sinais, linha a linha
    cb.value = var in candidatas
for var, (sinal, lag) in candidatas.items():
    ui._sign_tgs[var].value = sinal           # +1 piora / −1 melhora o parâmetro
    ui._sign_lags[var].value = lag

print("candidatas:", ui.candidates())
print("sinais esperados:", ui.expected_signs())
print("especificação corrente:", ui.current_spec().describe())

In [ ]:
ui.btn_fit_now.click()              # = clicar "Ajustar agora"

fit = ui.fit_
print(f"{fit.model_name} · n={fit.nobs} · AIC={fit.aic:.1f} · BIC={fit.bic:.1f} "
      f"· R²={fit.rsquared:.3f}")
fit.coef_frame()                    # na tela, com estrelas de significância e a coluna coerência


> **Ajuste desatualizado.** Qualquer mexida nos controles da especificação marca o ajuste como
> desatualizado (`ui._dirty_since_fit`) e **invalida** o que dependia dele — diagnóstico, projeção
> e backtest saem da tela com o aviso do motivo. É de propósito: o pior erro possível é ler um
> diagnóstico de um modelo e uma projeção de outro.


## 5. Aba ③ **Seleção** — a busca champion-challenger

**O que olhar.**

1. **O tamanho da grade, antes de rodar.** A busca é cara: cada especificação é ajustada na amostra
   cheia e, se passar nos filtros duros, revalidada **janela a janela**. O total cresce com o
   conjunto de defasagens **elevado** ao número de variáveis. O cartão do topo conta quantas
   especificações e quantos ajustes isso custa — leia antes de esperar.
2. **O ranking.** ★ marca a campeã pelo critério. As linhas de **benchmark** (ARIMA e ingênuos)
   entram na mesma tabela de propósito: um modelo macro que não bate o ARIMA **fora da amostra**
   ainda não está pronto, por melhor que seja o R². `vs ARIMA` abaixo de 1 = erro menor que o dele.
3. **As descartadas — e por quê.** Duas causas: **sinal invertido** (desqualificador) e **VIF
   acima do teto** (duas macro explicando a mesma coisa). Ver o que saiu importa tanto quanto ver a
   campeã: se quase tudo caiu por sinal, o problema costuma estar no **sinal declarado** ou na
   transformação da variável, não nas especificações.

**A decisão.** O critério é uma régua, não um veredito. Diferenças mínimas de RMSE não decidem
nada, e a especificação mais defensável costuma ser a mais **parcimoniosa** entre as equivalentes —
por isso a tela deixa você **adotar outra**, inclusive uma descartada, se discorda do sinal que
declarou. A adotada passa a ser o **modelo vigente** nas abas seguintes.

In [ ]:
# = ajustar a grade (aba ② Especificação) e conferi-la (aba ③ Seleção)
ui.tx_lag_set.value = "0,1"      # defasagens que a BUSCA varre (a matriz de sinais é do ajuste único)
ui.tx_ar_orders.value = "1"
ui.sl_max_vars.value = 2         # até 2 macro por especificação
ui.fl_vif_max.value = 5.0
ui.dd_criterion.value = "oos_rmse"
ui.sl_horizon.value = 6          # horizonte da validação walk-forward
ui.sl_min_train.value = 72       # janela mínima de treino

ui.btn_grid_size.click()         # = clicar "Conferir a grade"
ui._grid_size()                  # candidatas, specs, janelas e o total de ajustes

In [ ]:
ui.cb_require_signs.value = True     # filtro duro de sinal econômico ligado
ui.cb_include_bench.value = True     # ARIMA e ingênuos na mesma régua
ui.cb_cobertura.value = False        # medir cobertura aqui é bem mais lento (há aba própria)
ui.btn_search.click()                # = clicar "Rodar busca"

print("campeã adotada:", ui.selected_spec_.describe())
ui.search_.top(8)[["modelo", "status", "n_vars", "AIC", "max_vif", "oos_rmse", "vs_arima"]]

In [ ]:
# O seletor de escolha manual: campeã, demais qualificadas e as descartadas (marcadas ⚠)
for rotulo in [r for r, _ in ui.dd_pick_spec.options][:6]:
    print(" •", rotulo)

# Para discordar da campeã, é isto — na tela, o dropdown + "Adotar e ajustar":
#   ui.dd_pick_spec.value = "<describe() da especificação>"
#   ui.btn_pick_fit.click()


**A vantagem sobre os benchmarks é real?** O botão *Comparar com os benchmarks* revalida a
especificação adotada e roda **Diebold-Mariano** contra cada referência. H0: **mesma acurácia** — p
pequeno significa que a diferença não é ruído amostral. RMSE menor **sem** p pequeno é o caso mais
comum em série curta: a campeã parece melhor, mas você não consegue distingui-la do passeio
aleatório — e aí a parcimônia decide.

In [ ]:
ui.btn_dm.click()      # = clicar "Comparar com os benchmarks"
ui.compare_


## 6. Aba ④ **Diagnóstico** — o que o ajuste deixou no resíduo

**O que olhar.** O **placar por família de teste** resume tudo: autocorrelação (Ljung-Box,
Breusch-Godfrey), heterocedasticidade (Breusch-Pagan, White, ARCH-LM), normalidade (Jarque-Bera),
estabilidade (CUSUM, Chow, Quandt-Andrews) e colinearidade (VIF). Cada bloco mostra o **pior**
veredito da família e a evidência. ⚠️ significa **inconclusivo** (o teste não rodou ou não tem
graus de liberdade), não aprovação. Abaixo vêm a tabela completa — em que a coluna **ok** já
traduz todos os testes para a mesma direção, ✓ = o resultado desejável — e o cartão *O que fazer*.

**A decisão — quando um teste falha.** Nenhum teste sozinho reprova um modelo; a leitura é
conjunta. Na prática:

| Falhou | O que costuma significar | O que fazer |
|---|---|---|
| **Autocorrelação** | sobrou dinâmica que o modelo não capturou | aumentar a ordem AR, revisar as defasagens das macro |
| **Heterocedasticidade** | variância do erro muda com o ciclo | covariância **HAC (Newey-West)** nos erros-padrão; os coeficientes seguem válidos |
| **Normalidade** | caudas grossas, quase sempre por um evento pontual | não é fatal para a média; afeta a **banda** — confira na aba ⑥ Backtest |
| **Estabilidade** | os coeficientes mudaram no meio da amostra | é o mais grave para **projetar**: encurtar a amostra, tratar o evento, ou aceitar que a relação quebrou |
| **VIF** | duas macro explicando a mesma coisa | tirar uma das duas — o sinal da que fica vira por acaso amostral |

A **estabilidade** é a que mais pesa: projetar assume que a relação estimada continua valendo.

In [ ]:
# = clicar "Rodar diagnóstico" na aba ④ Diagnóstico
ui.fl_alpha_diag.value = 0.05
ui.cb_diag_plots.value = True          # ajuste observado × previsto + painel de resíduos
ui.btn_diag.click()

for b in ui.diag_blocks_:              # o placar, bloco a bloco
    print(f" {b['veredito']:<14} {b['bloco']}")

In [ ]:
ui.diagnostics_[["teste", "estatistica", "p_valor", "ok", "conclusao"]]


> **Lendo o resultado acima.** A série de referência tem uma recessão e um evento embutidos — e é
> exatamente por isso que **normalidade** e **estabilidade** reprovam: o evento deixa caudas grossas
> e desloca os coeficientes no meio da amostra. Num estudo de verdade a conduta seria a da tabela:
> erros-padrão **HAC** para a heterocedasticidade, tratar o evento (encurtar a amostra ou usar
> `Specification(events=...)` pela API) para a estabilidade, e depois checar na aba ⑥ se a banda
> ainda cobre o que promete. Reprovar aqui não impede projetar — impede projetar **sem dizer isso**.


## 7. Aba ⑤ **Cenários & Projeção** — o produto final

O **cenário** é a trajetória futura das macro; o modelo vigente a traduz na trajetória do
parâmetro. São **três caminhos** para montá-lo, e eles não competem:

1. **Cenários padrão (um clique).** Base por reversão à média + adverso ($+2$ desvios-padrão na
   variável de estresse) e otimista ($-1$ desvio). Serve para **ver a sensibilidade na hora** — não
   substitui o cenário oficial.
2. **Choque parametrizado.** Um choque aditivo sobre a base, com **magnitude** (em desvios-padrão,
   comparável entre variáveis de escalas diferentes, ou em unidades) e **persistência**: `1,0`
   mantém o choque até o fim (deslocamento permanente) e valores menores o fazem decair
   geometricamente (choque temporário).
3. **Colar a trajetória da área econômica.** Cabeçalho com os nomes das colunas da macro e uma
   linha por período, separados por TAB (colagem direta do Excel), vírgula ou ponto e vírgula.
   Colunas de data são ignoradas (o calendário é reconstruído), variáveis que faltarem são
   completadas pela base — e a tela diz quais. É assim que o cenário oficial entra: sem redigitar,
   sem planilha paralela e com o **mesmo motor de projeção** dos demais.

**Lendo o leque.** A **linha** é a trajetória central de cada cenário; a **faixa** é o intervalo do
$\alpha$ escolhido ($\alpha = 0{,}10$ ⇒ 90%), obtido por reamostragem dos resíduos propagada pela
dinâmica do modelo — a incerteza **acumula** ao longo do horizonte. Leia os dois juntos: a
distância **entre** cenários mede o efeito do ciclo; a **largura** da faixa mede a incerteza do
modelo. Quando a faixa do base engole o adverso, o cenário não está dizendo mais do que o ruído do
próprio ajuste — e essa é uma conclusão legítima, que vale registrar.

A **projeção ponderada** é a curva única que segue para o processo seguinte: a média das
trajetórias pelos pesos declarados. Ela existe para que não haja duas respostas para a mesma
pergunta em áreas diferentes.

In [ ]:
# Parâmetros comuns da projeção (topo da aba ⑤)
ui.sl_scen_horizon.value = 12          # 12 meses à frente
ui.fl_scen_alpha.value = 0.10          # banda de 90%
ui.sl_scen_sims.value = 500            # simulações da banda
ui.dd_stress_var.value = "desemprego"  # a variável estressada nos cenários padrão
ui.tx_scen_probs.value = "0.5,0.3,0.2" # pesos: base, otimista, adverso

# Caminho 1 — "Montar base / adverso / otimista"
ui.btn_scen_padrao.click()
print("cenários:", ui.scenarios_.names(), "| pesos:", ui.scenarios_.probabilities())

In [ ]:
# Caminho 2 — choque parametrizado (substitui o conjunto anterior)
ui.dd_scen_base.value = "revert"       # base: reversão à média (há hold e trend)
ui.fl_shock_mag.value = 2.0            # +2 desvios-padrão no desemprego
ui.dd_shock_unit.value = "sd"
ui.fl_shock_persist.value = 0.85       # < 1 ⇒ choque temporário, decaindo
ui.fl_shock_otim.value = 0.5           # otimista = metade do choque, no sentido oposto
ui.btn_scen_choque.click()

desvio = (ui.scenarios_.get("adverso").macro["desemprego"]
          - ui.scenarios_.get("base").macro["desemprego"])
print("desvio do adverso sobre a base (p.p. de desemprego):")
print(desvio.round(3).to_string())

In [ ]:
# Caminho 3 — colar a trajetória que veio da área econômica.
# "Gerar modelo para colar" preenche a caixa com a trajetória base no formato certo:
ui.btn_scen_modelo.click()
print("gabarito (3 primeiras linhas):")
print("\n".join(ui.ta_scen_paste.value.splitlines()[:3]), "\n")

# ...e no lugar dele entra o que você colou do Excel (uma linha por período do horizonte).
# Colunas ausentes são completadas pela base — aqui só o desemprego veio da área econômica.
h = int(ui.sl_scen_horizon.value)
ui.ta_scen_paste.value = "desemprego\n" + "\n".join(f"{9.4 + 0.12 * i:.2f}" for i in range(h))
ui.tx_scen_nome.value = "economia_oficial"
ui.fl_scen_peso.value = 0.20           # peso declarado; o conjunto é renormalizado para somar 1
ui.btn_scen_add.click()                # = clicar "Adicionar cenário"

print("cenários montados:", ui.scenarios_.names())
print("pesos:", {k: round(v, 3) for k, v in ui.scenarios_.probabilities().items()})

In [ ]:
ui.cb_proj_plot.value = True
ui.btn_project.click()                 # = clicar "Projetar"

proj = ui.projection_
print(f"horizonte={proj.horizon} · α={proj.alpha} · cenários={list(proj.paths)}")
proj.mean_frame().round(4).head()      # trajetória central de cada cenário

In [ ]:
# A curva única (média ponderada pelos pesos) — o que segue para o processo seguinte
ui.weighted_.round(4).to_frame("PD projetada (ponderada)").head()


## 8. Aba ⑥ **Backtest** — a banda cobre o que promete?

**O que olhar.** O erro fora da amostra da aba ③ mede a **trajetória central**. Aqui se testa a
outra metade da projeção: a **banda**. O procedimento reestima o modelo janela a janela, projeta o
horizonte com a macro **efetivamente observada** (sem ver o futuro) e conta quantas vezes o
realizado caiu fora do intervalo.

- **Erro por horizonte.** O erro **cresce** com o passo à frente — é assim que deve ser. Interessa
  o formato: um salto abrupto num passo específico costuma indicar sazonalidade não modelada ou
  defasagem mal escolhida. O **viés** revela projeção sistematicamente otimista ou pessimista.
- **Cobertura.** Uma banda de 90% honesta erra ~10% das vezes. Errar muito menos é
  **conservadorismo caro**; errar muito mais é **subestimar a incerteza**.
- **Kupiec (POF).** H0: a taxa de violação é a nominal. p pequeno **rejeita** ⇒ a banda tem
  largura errada.
- **Christoffersen.** H0: as violações são **independentes** no tempo. p pequeno indica violações
  em *cluster*: a banda até acerta na média, mas falha justamente quando o ciclo vira — o pior
  momento possível.
- **O gráfico realizado × previsto**, janela a janela, destaca as violações: vale olhar *quando*
  elas acontecem, não só quantas.

**A decisão.** Passar nos dois testes é o que sustenta o intervalo perante a validação
independente. Se Kupiec reprova por banda estreita, o caminho usual é revisar a fonte de incerteza
(mais simulações não resolvem — o problema é o modelo de resíduo). Se Christoffersen reprova, a
banda não está capturando a mudança de regime: reveja estabilidade na aba ④ antes de mexer aqui.

> É a ação mais cara da interface — cada janela é um reajuste completo mais uma projeção simulada.
> O cartão do topo conta as janelas antes de rodar.

In [ ]:
# = configurar e clicar "Rodar backtest" na aba ⑥
ui.sl_bt_min_train.value = 72
ui.sl_bt_horizon.value = 3
ui.sl_bt_step.value = 1
ui.fl_bt_alpha.value = 0.10
ui.sl_bt_sims.value = 200

ui.btn_bt_info.click()          # = "Conferir as janelas"
print(ui._bt_params())

In [ ]:
ui.btn_backtest.click()

wf = ui.backtest_
print(f"janelas={wf['n_windows']} · RMSE={wf['rmse']:.4f} · MAE={wf['mae']:.4f}")
ui.coverage_[["passo", "n", "violacoes", "nominal", "cobertura",
              "kupiec_pvalue", "christoffersen_pvalue", "ok"]]


## 9. Aba ⑦ **Exportar** — o que sai da sessão

O primeiro cartão é um **placar do que já está pronto** (série, modelo vigente, busca, diagnóstico,
projeção, backtest): tudo o que sai daqui leva o **estado corrente**, e o que estiver faltando
simplesmente não vai junto. Quatro saídas:

- **Relatório HTML** — documento autocontido (figuras embutidas) com especificação, coeficientes,
  métricas, bateria de diagnóstico e o leque. É o anexo que acompanha o modelo na validação
  independente. Se o arquivo já existir, o botão pede confirmação antes de sobrescrever.
- **MLflow** — versiona o estudo: parâmetros, métricas, diagnóstico, ranking, figuras e o relatório
  como artefato. Deixe o experimento vazio para usar o ativo da sessão.
- **JSON da configuração** — a `StudyConfig`, o mesmo objeto que o pipeline declarativo (`run_study`)
  consome fora do notebook. É o que torna a projeção **reproduzível**: candidatas, sinais,
  defasagens, critério, teto de VIF, horizonte, variável de estresse e pesos. *Carregar* repõe
  todos os controles da tela.
- **Tabelas em CSV** — projeção (formato longo), ranking com o motivo de cada descarte,
  diagnóstico, cobertura, coeficientes e estacionariedade. Há formato com **vírgula decimal** para
  abrir direto no Excel em pt-BR; e cada tabela também aparece numa caixa de texto para copiar.

In [ ]:
import tempfile, pathlib

tmp = pathlib.Path(tempfile.mkdtemp())          # no seu caso, um caminho de verdade

ui.tx_report_path.value = str(tmp / "relatorio_pd.html")
ui.btn_report.click()                            # = "Gerar relatório HTML"
print("relatório:", ui.report_path_,
      "|", pathlib.Path(ui.report_path_).stat().st_size // 1024, "KB")

In [ ]:
ui.btn_cfg_show.click()                          # = "Ver JSON da sessão"
print("\n".join(ui.ta_config_json.value.splitlines()[:14]), "\n...")

# Salvar / carregar (na tela são os botões ao lado do campo de arquivo):
ui.tx_cfg_path.value = str(tmp / "pd_segmento_referencia.json")
ui.btn_cfg_save.click()
print("configuração salva em:", ui.tx_cfg_path.value)

In [ ]:
# Tabelas do estudo: escolha a tabela e o formato, e ela sai pronta para copiar/salvar
ui.dd_exp_tabela.value = "projecao"
ui.dd_exp_fmt.value = "csv_br"                   # vírgula decimal, ponto e vírgula (Excel pt-BR)
ui.btn_exp_mostrar.click()
print("\n".join(ui.ta_export_tabela.value.splitlines()[:4]))

ui.tx_exp_path.value = str(tmp / "projecao.csv")
ui.btn_exp_salvar.click()                        # = "Salvar CSV"
print("\narquivo:", ui.tx_exp_path.value)

In [ ]:
# MLflow — aqui apontado para um tracking local temporário só para o tutorial rodar.
# Num ambiente com tracking configurado (Databricks, por exemplo), basta clicar no botão.
import mlflow

mlflow.set_tracking_uri((tmp / "mlruns").as_uri())
ui.tx_mlflow_exp.value = ""                      # vazio = experimento ativo da sessão
ui.tx_mlflow_run.value = "satelite_pd_tutorial"
ui.btn_mlflow.click()                            # = "Registrar no MLflow"

run = mlflow.get_run(ui.mlflow_run_id_)
print("run_id:", ui.mlflow_run_id_[:12])
print("métricas:", {k: round(v, 4) for k, v in sorted(run.data.metrics.items())[:6]})


## 10. O atalho: **estudo completo em um clique**

No fim da aba ② Especificação há o botão **Rodar estudo completo**. Ele faz exatamente as cinco
chamadas do fluxo manual, na ordem: **busca** champion-challenger sobre a grade → **ajuste** da
campeã → **bateria de diagnóstico** → **cenários padrão** sobre a variável de estresse →
**projeção condicional**. Ao terminar, as abas ③, ④ e ⑤ aparecem preenchidas, com uma tabela de
progresso por etapa e o tempo decorrido. O **backtest fica de fora de propósito**: é caro e tem aba
própria.

**Quando usar cada um.**

- **Um clique** quando você já sabe o que quer: reprocessamento periódico do mesmo segmento, uma
  segunda opinião rápida, ou o primeiro passe para ver se há sinal macro ali.
- **Passo a passo** quando a decisão ainda não está tomada — e ela raramente está na primeira
  rodada: escolher entre especificações equivalentes, discordar da campeã, tratar um evento, montar
  o cenário oficial em vez do padrão.

Nada fica trancado: depois do clique você pode discordar de qualquer etapa na aba correspondente
(adotar outra especificação, trocar o cenário, reprojetar).

Abaixo abrimos uma **segunda** interface sobre os mesmos dados e a alimentamos com a configuração
da primeira (`to_config()` → `from_config()`) — de quebra, é a demonstração de como uma
configuração versionada volta para a tela.

In [ ]:
ui2 = SatelliteUI(series=taxa, macro=macro, kind="pd",
                  segment="segmento_referencia", problem_label="PD")
ui2.from_config(ui.to_config())          # repõe candidatas, sinais, grade, critério, cenário…
ui2.tx_nome.value = "pd_um_clique"
ui2.cb_study_report.value = True         # gera também o HTML do relatório

ui2.btn_run_study.click()                # = clicar "Rodar estudo completo"
ui2.study_.summary()

In [ ]:
# O StudyResult tem tudo o que as abas mostraram — e a interface ficou preenchida:
res = ui2.study_
print("busca      :", len(res.search.ranking), "linhas no ranking |",
      "campeã:", res.search.best_spec.describe())
print("ajuste     :", res.fit.model_name, "· AIC", round(res.fit.aic, 1))
print("cenários   :", res.scenarios.names())
print("projeção   :", res.projection.horizon, "períodos · α =", res.projection.alpha)
print("relatório  :", len(res.report_html or ""), "bytes de HTML")
print("backtest   :", ui2.backtest_, "(de propósito: tem aba própria)")


## 11. Recuperando o resultado **no notebook**

A ponte entre a tela e o pipeline: tudo o que a interface produziu fica como **atributo do objeto**,
nos tipos da própria biblioteca (os mesmos do tutorial 09). Nada de reimplementar — é só pegar.

| No objeto | O que é | De onde veio |
|---|---|---|
| `ui.series`, `ui.macro` | `RiskSeries` e `DataFrame` correntes | aba ① (ou `ui.set_data(...)`) |
| `ui.stationarity_` | tabela ADF/KPSS/PP com a ordem I(d) | aba ① |
| `ui.fit_`, `ui.model_` | `FitResult` e o modelo vigente | aba ② ou ③ |
| `ui.search_`, `ui.selected_spec_` | `SearchResult` e a `Specification` adotada | aba ③ |
| `ui.compare_` | comparação vs benchmarks com Diebold-Mariano | aba ③ |
| `ui.diagnostics_`, `ui.diag_blocks_`, `ui.vif_` | bateria completa, placar e VIF | aba ④ |
| `ui.scenarios_`, `ui.projection_`, `ui.weighted_` | `ScenarioSet`, `Projection` e a curva única | aba ⑤ |
| `ui.projection_frame()` | a projeção em **formato longo**, com a linha `ponderado` | aba ⑤ |
| `ui.backtest_`, `ui.coverage_` | dicionário do backtest e a tabela de cobertura | aba ⑥ |
| `ui.study_` | `StudyResult` do estudo em um clique | aba ② |
| `ui.to_config()` / `ui.from_config(cfg)` | a `StudyConfig` da tela, ida e volta | aba ⑦ |
| `ui.report_path_`, `ui.mlflow_run_id_` | último relatório gravado e último *run* | aba ⑦ |

Também são públicos `ui.candidates()`, `ui.expected_signs()`, `ui.lag_por_variavel()`,
`ui.current_spec()`, `ui.build_model()` (o modelo **não** ajustado da tela) e `ui.set_data(serie,
macro, kind=...)`, que troca os dados e zera o que pertencia aos antigos.

> **Dois horizontes, de propósito.** O `horizon` da `StudyConfig` é o da **validação** (aba ②,
> usado no walk-forward e no estudo em um clique); a aba ⑤ tem o **seu** horizonte de projeção.
> Validar em 6 passos e projetar 12 meses é comum — e não são a mesma decisão.

In [ ]:
print("modelo vigente :", type(ui.model_).__name__, "|", ui.selected_spec_.describe())
print("diagnóstico    :", ui.diagnostics_.shape[0], "testes |",
      int(ui.diagnostics_["ok"].fillna(False).sum()), "com resultado desejável")
print("cenários       :", ui.scenarios_.names())
print("backtest       :", ui.backtest_["n_windows"], "janelas · RMSE",
      round(ui.backtest_["rmse"], 4))

# A saída que segue para o processo seguinte (uma linha por cenário e período):
proj_longa = ui.projection_frame()
proj_longa.head()

In [ ]:
# E a configuração que reproduz tudo isso fora do notebook:
cfg = ui.to_config()
print(cfg.name, "|", cfg.model, "| candidatas:", cfg.candidates)
print("sinais:", cfg.expected_signs, "| horizonte:", cfg.horizon,
      "| estresse:", cfg.stress_var, "| pesos:", cfg.scenario_probabilities)

# fora da tela, a mesma configuração roda o estudo inteiro:
#   res = E.run_study(cfg, ui.series, ui.macro, make_report=True)
cfg.to_dict()                    # é este dicionário que vira o JSON da aba ⑦


## 12. Fechamento — limites e para onde ir

**O que a interface não faz** (de propósito, e é bom saber antes de procurar o botão):

- **um segmento por vez.** A tela modela **uma** série. Painel de segmentos (`PanelSatellite`),
  VAR/VECM e cointegração ficam na API — o dropdown de modelos traz apenas os que ajustam,
  preveem e projetam pela mesma interface.
- **sem dummies de evento na tela.** `Specification(events={"covid": ("2020-03", "2020-08")})`
  existe na API; a interface trata evento pela escolha da amostra e pela leitura da estabilidade
  no diagnóstico.
- **a busca usa uma defasagem por variável** e não mexe na matriz de sinais: a especificação
  adotada é independente dos controles do ajuste único (decisão deliberada, avisada na própria
  tela).
- **todos os cenários compartilham o mesmo calendário e horizonte** — a colagem exige exatamente o
  número de linhas do horizonte.
- **o backtest não entra no estudo em um clique**, e o cálculo de cobertura na busca fica desligado
  por padrão: nos dois casos por custo.
- **relatório e MLflow escrevem em arquivo/tracking** — no Databricks, use um caminho persistente
  (DBFS/Volumes); o diretório do driver é efêmero.

**Para onde ir depois:**

- [`09_tutorial_modelos_econometricos.ipynb`](09_tutorial_modelos_econometricos.ipynb) — a **API**
  por trás de cada botão: transformações, ARDL, fator $Z$, beta/fractional, VAR/VECM, painel,
  `search`, cenários e `run_study`.
- [`08_tutorial_capital_economico.ipynb`](08_tutorial_capital_economico.ipynb) — o uso do fator
  prospectivo em **estresse e capital**: o fator $Z$ estimado aqui é o mesmo que alimenta a
  simulação multifatorial.
- [`05_tutorial_instalacao_e_interfaces.ipynb`](05_tutorial_instalacao_e_interfaces.ipynb) — as
  outras interfaces da casa (árvore e construtor de modelos), que cobrem o **eixo transversal**:
  o modelo transversal ordena o risco entre clientes; o satélite desloca o nível da curva conforme
  o ciclo.